# Breast Cancer Classification using Logistic Regression

**Dataset:** `breast_cancer.csv`  
**Model:** Logistic Regression  
**Interfaces:** Gradio and Flask

This dataset uses `Class = 2` for benign and `Class = 4` for malignant. The notebook maps these to 0 and 1 for binary classification.

**Disclaimer:** This is an educational machine-learning project, not a medical diagnostic tool.


## 1. Install required libraries

In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn gradio flask pyngrok

## 2. Upload and read the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select breast_cancer.csv

import pandas as pd
import numpy as np

csv_name = next(name for name in uploaded if name.lower().endswith(".csv"))
df = pd.read_csv(csv_name)

print("Dataset shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())

## 3. Clean the dataset and prepare features

In [ ]:
target_column = "Class"
df = df.dropna(subset=[target_column]).copy()

# In this dataset, 2 = benign and 4 = malignant.
df[target_column] = pd.to_numeric(df[target_column], errors="coerce")
df = df.dropna(subset=[target_column])
df = df[df[target_column].isin([2, 4])].copy()
df[target_column] = df[target_column].map({2: 0, 4: 1}).astype(int)

X = df.drop(columns=[target_column]).apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median())
y = df[target_column]

feature_names = X.columns.tolist()
feature_medians = X.median().to_dict()

print("Features:", feature_names)
print("Class counts (0=Benign, 1=Malignant):")
print(y.value_counts().sort_index())

## 4. Split data and train Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)
print("Logistic Regression model trained successfully!")

## 5. Evaluate the model

In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)
import matplotlib.pyplot as plt

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print("\\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Benign (Class 2)", "Malignant (Class 4)"]
))
print("Confusion matrix:\\n", confusion_matrix(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Benign (Class 2)", "Malignant (Class 4)"]
)
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()

## 6. Create a prediction function

In [ ]:
def predict_cancer(values):
    row = pd.DataFrame([values], columns=feature_names)
    prediction = int(model.predict(row)[0])
    probabilities = model.predict_proba(row)[0]

    label = "Malignant (Class 4)" if prediction == 1 else "Benign (Class 2)"
    confidence = float(probabilities[prediction])
    return label, confidence

# Test with the first row
print(predict_cancer(X.iloc[0].tolist()))

## 7. Build the Gradio interface

In [ ]:
import gradio as gr

def gradio_predict(*values):
    label, confidence = predict_cancer(values)
    return (
        f"Predicted dataset label: {label}\\n"
        f"Model confidence: {confidence:.2%}\\n\\n"
        "Educational demonstration only — not a medical diagnosis."
    )

gr_inputs = [
    gr.Number(label=name, value=float(feature_medians[name]))
    for name in feature_names
]

with gr.Blocks() as demo:
    gr.Markdown("# Breast Cancer Classification")
    gr.Markdown(
        "Logistic Regression using nine numerical features. "
        "Educational demonstration only; not a medical diagnostic tool."
    )
    result_box = gr.Textbox(label="Prediction result", lines=4)
    predict_button = gr.Button("Predict")
    predict_button.click(
        fn=gradio_predict, inputs=gr_inputs, outputs=result_box
    )

demo.launch(share=True)

## 8. Build the Flask interface

In [ ]:
from flask import Flask, request, render_template_string
from pyngrok import ngrok
from getpass import getpass
import threading
import time

token = getpass("Enter your ngrok auth token: ")
ngrok.set_auth_token(token.strip())

app = Flask(__name__)

HTML = """
<!doctype html>
<html>
<head>
  <title>Breast Cancer Classification</title>
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <style>
    body { font-family: Arial, sans-serif; max-width: 1050px; margin: 30px auto; padding: 0 18px; }
    .grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 12px; }
    label { display: block; font-size: 14px; margin-bottom: 5px; }
    input { width: 100%; box-sizing: border-box; padding: 8px; }
    button { margin-top: 18px; padding: 10px 18px; cursor: pointer; }
    .result { background: #f1f1f1; padding: 14px; margin: 18px 0; white-space: pre-line; }
    .notice { color: #8a1c1c; }
  </style>
</head>
<body>
  <h1>Breast Cancer Classification</h1>
  <p>Logistic Regression using the breast cancer dataset.</p>
  <p class="notice"><b>Educational use only — not a medical diagnostic tool.</b></p>
  {% if result %}<div class="result">{{ result }}</div>{% endif %}
  <form method="post">
    <div class="grid">
    {% for name in feature_names %}
      <div>
        <label for="{{ name }}">{{ name }}</label>
        <input type="number" step="any" id="{{ name }}" name="{{ name }}"
               value="{{ defaults[name] }}" required>
      </div>
    {% endfor %}
    </div>
    <button type="submit">Predict</button>
  </form>
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def home():
    result = ""
    if request.method == "POST":
        try:
            values = [float(request.form[name]) for name in feature_names]
            label, confidence = predict_cancer(values)
            result = (
                f"Predicted dataset label: {label}\n"
                f"Model confidence: {confidence:.2%}\n\n"
                "Educational demonstration only — not a medical diagnosis."
            )
        except (ValueError, KeyError):
            result = "Please enter valid numbers for every feature."

    return render_template_string(
        HTML,
        feature_names=feature_names,
        defaults={name: float(feature_medians[name]) for name in feature_names},
        result=result,
    )

def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(2)
public_url = ngrok.connect(5000).public_url
print("Your Flask app URL:", public_url)


## Files for GitHub
Upload these files together:
- `Breast_Cancer_Logistic_Regression_Gradio_Flask.ipynb`
- `breast_cancer.csv`
- `README.md`

For Flask in Google Colab, the notebook asks for an ngrok authentication token. Keep this token private.

**Disclaimer:** This model is for coursework only and must not be used to make healthcare decisions.
